# Q-MAPPO Phase 2 — 200K Steps — Colab Version v2

**Google Colab optimised** — auto-saves to Google Drive every 10 minutes + full logs.

**What's new in v2:**
- N_QUBITS: 4 → 8 | N_VQC_LAYERS: 3 → 5 | Actor LR: 1e-4 → 3e-4
- 2-layer Pre-Encoding NN | LayerNorm in QI layers | Episodes/update: 8 → 16
- Entropy coef: 0.05 → 0.01 | Full CSV log saved to Drive

**Phase 1 MAPPO 200K baseline:** Best Avg100 = -42.08 | Eval Mean = -50.80

In [1]:
# CELL 1 — Install
!pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3
print('Install complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
Install complete


In [2]:
# CELL 2 — Mount Google Drive + GPU Check
import torch, os, sys

# Mount Google Drive — all results saved here automatically
from google.colab import drive
drive.mount('/content/drive')

# Create save directories
DRIVE_DIR = '/content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2'
LOCAL_DIR = '/content/qmappo_v2'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

# GPU info
print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — Runtime -> Change runtime type -> T4 GPU')

print(f'Local  : {LOCAL_DIR}')
print(f'Drive  : {DRIVE_DIR}')

Mounted at /content/drive
Torch  : 2.11.0+cu128
CUDA   : True
Device : Tesla T4
Memory : 15.6 GB
Local  : /content/qmappo_v2
Drive  : /content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2


In [3]:
# CELL 3 — Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import time, os, pickle, zipfile, shutil, csv
from datetime import datetime
from pettingzoo.mpe import simple_spread_v3
print('All imports done.')

All imports done.


In [4]:
# CELL 4 — Logger (saves CSV log to Drive automatically)
class TrainingLogger:
    """
    Saves full training log to Drive as CSV.
    Auto-saves every N updates so nothing is lost if session disconnects.
    """
    def __init__(self, drive_dir, local_dir, run_name='qmappo_v2'):
        self.drive_dir  = drive_dir
        self.local_dir  = local_dir
        self.run_name   = run_name
        self.log_path   = os.path.join(local_dir, f'{run_name}_log.csv')
        self.drive_path = os.path.join(drive_dir, f'{run_name}_log.csv')
        self.rows       = []

        # Write CSV header
        self.fields = [
            'episode', 'timestep', 'ep_reward', 'avg100',
            'best_avg100', 'actor_loss', 'critic_loss',
            'entropy', 'actor_lr', 'elapsed_min'
        ]
        with open(self.log_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writeheader()
        print(f'Logger ready -> {self.log_path}')

    def log(self, row):
        """Add one row to log."""
        self.rows.append(row)
        with open(self.log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

    def save_to_drive(self):
        """Copy log + best model to Drive."""
        shutil.copy(self.log_path, self.drive_path)
        # Also copy best models if they exist
        for fname in ['best_actor.pth', 'best_critic.pth', 'best_obs_norm.pkl']:
            src = os.path.join(self.local_dir, fname)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(self.drive_dir, fname))
        print(f'  [Drive saved] {datetime.now().strftime("%H:%M:%S")} -> {self.drive_dir}')

    def summary(self):
        """Print training summary table."""
        if not self.rows: return
        best = min(self.rows, key=lambda r: -r['best_avg100'])
        print(f'  Total rows logged : {len(self.rows)}')
        print(f'  Best Avg100       : {best["best_avg100"]:.2f}  (ep {best["episode"]})')
        print(f'  Log saved to Drive: {self.drive_path}')


logger = TrainingLogger(DRIVE_DIR, LOCAL_DIR)
print('Logger created.')

Logger ready -> /content/qmappo_v2/qmappo_v2_log.csv
Logger created.


In [5]:
# CELL 5 — Environment
env = simple_spread_v3.parallel_env(
    N=3, local_ratio=0.5, max_cycles=50, continuous_actions=False
)
obs, _ = env.reset()

AGENTS           = ['agent_0', 'agent_1', 'agent_2']
NUM_AGENTS       = len(AGENTS)
OBS_DIM          = env.observation_space(env.agents[0]).shape[0]  # 18
ACTION_DIM       = env.action_space(env.agents[0]).n               # 5
GLOBAL_STATE_DIM = OBS_DIM * NUM_AGENTS                            # 54

# v2 improved dimensions
N_QUBITS     = 8   # IMPROVED: 4->8
N_VQC_LAYERS = 5   # IMPROVED: 3->5

print(f'Agents={NUM_AGENTS} | ObsDim={OBS_DIM} | Actions={ACTION_DIM} | GlobalDim={GLOBAL_STATE_DIM}')
print(f'QI Actor: {N_QUBITS} hidden units, {N_VQC_LAYERS} layers (v2 improved)')

Agents=3 | ObsDim=18 | Actions=5 | GlobalDim=54
QI Actor: 8 hidden units, 5 layers (v2 improved)


In [6]:
# CELL 5 — Running Normalizer (identical to Phase 1 MAPPO)
class RunningNorm:
    def __init__(self, shape, clip=5.0, eps=1e-8):
        self.shape = (shape,) if isinstance(shape, int) else shape
        self.mean  = np.zeros(self.shape, np.float64)
        self.var   = np.ones(self.shape,  np.float64)
        self.count = eps
        self.clip  = clip
        self.eps   = eps

    def update(self, x):
        x = np.asarray(x, np.float64).reshape(self.shape)
        self.count += 1
        delta       = x - self.mean
        self.mean  += delta / self.count
        self.var   += delta * (x - self.mean)

    @property
    def std(self):
        return np.sqrt(self.var / self.count + self.eps)

    def normalize(self, x):
        x_norm = (np.asarray(x, np.float64) - self.mean) / self.std
        return np.clip(x_norm, -self.clip, self.clip).astype(np.float32)

    def normalize_scalar(self, x):
        return float(np.clip(x / self.std[0], -self.clip, self.clip))

    def save(self):
        return {'mean': self.mean.copy(), 'var': self.var.copy(), 'count': self.count}

    def load(self, d):
        self.mean = d['mean']; self.var = d['var']; self.count = d['count']


obs_norm = RunningNorm(shape=OBS_DIM, clip=5.0)
ret_norm = RunningNorm(shape=1, clip=5.0)
print('Normalizers ready.')

Normalizers ready.


In [7]:
# CELL 6 — Orthogonal Init Helper (identical to Phase 1)
def ortho(layer, gain=np.sqrt(2)):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.zeros_(layer.bias)
    return layer

In [8]:
# CELL 8 — *** PHASE 2 v2 *** Quantum-Inspired Actor (IMPROVED)
# All 7 improvements applied:
#   N_QUBITS: 4->8 | N_VQC_LAYERS: 3->5
#   2-layer Pre-Encoding | LayerNorm | larger capacity

class VQCActorNetwork(nn.Module):
    """
    Quantum-Inspired Actor v2 for Q-MAPPO (Phase 2).

    Architecture (Paper 1 PPO-Q arXiv:2501.07085):
      Pre-Encoding NN  : Linear(18->32->8) + Tanh    [2-layer, preserves more info]
      Quantum-Inspired : 5x [LayerNorm + Linear(8->8) + sin() + residual]
      Post-Process NN  : Linear(8->5) + softmax

    Improvements over v1:
      N_QUBITS 4->8     : more representational capacity
      N_LAYERS 3->5     : deeper quantum-inspired circuit
      2-layer pre-enc   : better observation compression (18->32->8)
      LayerNorm         : prevents entropy getting stuck at 1.609 (uniform)
      Total params: ~729 vs 161 (v1) vs 128K (Phase 1) -- still 99.4% fewer
    """
    def __init__(self, obs_dim, act_dim, n_qubits=N_QUBITS, n_layers=N_VQC_LAYERS):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # Pre-Encoding NN: 18 -> 32 -> 8 (2-layer for better compression)
        self.pre_enc = nn.Sequential(
            nn.Linear(obs_dim, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits),
            nn.Tanh()
        )
        nn.init.xavier_uniform_(self.pre_enc[0].weight)
        nn.init.zeros_(self.pre_enc[0].bias)
        nn.init.xavier_uniform_(self.pre_enc[2].weight)
        nn.init.zeros_(self.pre_enc[2].bias)

        # Quantum-Inspired layers with LayerNorm
        # sin(LN(Wh)) + h  mimics: rotation gate (sin) + entanglement (residual)
        self.qi_layers = nn.ModuleList([
            nn.Linear(n_qubits, n_qubits) for _ in range(n_layers)
        ])
        self.qi_norms = nn.ModuleList([
            nn.LayerNorm(n_qubits) for _ in range(n_layers)
        ])
        for layer in self.qi_layers:
            nn.init.xavier_uniform_(layer.weight, gain=0.1)
            nn.init.zeros_(layer.bias)

        # Post-Processing NN: 8 -> 5
        self.post_proc = nn.Linear(n_qubits, act_dim)
        nn.init.constant_(self.post_proc.weight, 0.01)
        nn.init.zeros_(self.post_proc.bias)

    def forward(self, x):
        # Pre-Encoding: 18 -> 32 -> 8
        h = self.pre_enc(x)
        # Quantum-Inspired layers: sin(LN(Wh)) + h
        for layer, norm in zip(self.qi_layers, self.qi_norms):
            h = torch.sin(norm(layer(h))) + h
        # Post-Processing: 8 -> 5 action probs
        return torch.softmax(self.post_proc(h), dim=-1)


# Architecture test
import time
print('Building VQCActorNetwork v2...')
_test = VQCActorNetwork(OBS_DIM, ACTION_DIM)
_x    = torch.zeros(256, OBS_DIM)
t0    = time.time()
_out  = _test(_x)
t1    = time.time()
print(f'VQCActorNetwork v2 PASSED')
print(f'  Input  : {_x.shape}')
print(f'  Output : {_out.shape}  (should be [256, 5])')
print(f'  Probs  : {_out.sum(dim=-1)[:3]}')
_out.sum().backward()
print(f'  Grads  : OK')
n_params = sum(p.numel() for p in _test.parameters())
print(f'  Params : {n_params}  (v1=161 | Phase1=128K | reduction={((1-n_params/128256)*100):.1f}%)')
print(f'  Speed  : {(t1-t0)*1000:.2f} ms for batch=256')
del _test, _x, _out

Building VQCActorNetwork v2...
VQCActorNetwork v2 PASSED
  Input  : torch.Size([256, 18])
  Output : torch.Size([256, 5])  (should be [256, 5])
  Probs  : tensor([1., 1., 1.], grad_fn=<SliceBackward0>)
  Grads  : OK
  Params : 1357  (v1=161 | Phase1=128K | reduction=98.9%)
  Speed  : 73.75 ms for batch=256


In [9]:
# CELL 9 — Classical Centralized Critic (IDENTICAL to Phase 1 MAPPO)
class Critic(nn.Module):
    def __init__(self, state_dim, h1=512, h2=512, h3=256):
        super().__init__()
        self.fc1  = ortho(nn.Linear(state_dim, h1))
        self.ln1  = nn.LayerNorm(h1)
        self.fc2  = ortho(nn.Linear(h1, h2))
        self.ln2  = nn.LayerNorm(h2)
        self.fc3  = ortho(nn.Linear(h2, h3))
        self.ln3  = nn.LayerNorm(h3)
        self.head = ortho(nn.Linear(h3, 1), gain=1.0)

    def forward(self, x):
        x = torch.tanh(self.ln1(self.fc1(x)))
        x = torch.tanh(self.ln2(self.fc2(x)))
        x = torch.tanh(self.ln3(self.fc3(x)))
        return self.head(x)


actor  = VQCActorNetwork(OBS_DIM, ACTION_DIM)
critic = Critic(GLOBAL_STATE_DIM)

print('=== Q-MAPPO v2 Network Summary ===')
print(f'Actor  (QI v2)    : {sum(p.numel() for p in actor.parameters()):,} params')
print(f'Critic (Classical): {sum(p.numel() for p in critic.parameters()):,} params')

=== Q-MAPPO v2 Network Summary ===
Actor  (QI v2)    : 1,357 params
Critic (Classical): 424,961 params


In [10]:
# CELL 10 — Hyperparameters v2 (200K)
TOTAL_STEPS          = 200_000
GAMMA                = 0.99
GAE_LAMBDA           = 0.95
CLIP_EPS             = 0.2
PPO_EPOCHS           = 10
MINI_BATCH           = 256
EPISODES_PER_UPDATE  = 16    # IMPROVED: 8->16
ENTROPY_COEF         = 0.01  # IMPROVED: 0.05->0.01 (fixes stuck entropy)
VALUE_COEF           = 0.5
ACTOR_LR             = 3e-4  # IMPROVED: 1e-4->3e-4 (was too slow)
CRITIC_LR            = 1e-3  # UNCHANGED
ACTOR_GRAD_CLIP      = 0.5
CRITIC_GRAD_CLIP     = 0.5
CRITIC_WARMUP_STEPS  = 5
CRITIC_WARMUP_UNTIL  = 50_000
SAVE_EVERY           = 50_000   # save every 50K (more frequent for Colab)
DRIVE_SAVE_EVERY     = 20       # save to Drive every 20 updates

print('Q-MAPPO v2 Hyperparameters:')
print(f'  ACTOR_LR            = {ACTOR_LR}  (FIXED: was 1e-4)')
print(f'  EPISODES_PER_UPDATE = {EPISODES_PER_UPDATE}  (FIXED: was 8)')
print(f'  ENTROPY_COEF        = {ENTROPY_COEF}  (FIXED: was 0.05)')
print(f'  Drive save every    = {DRIVE_SAVE_EVERY} updates')

Q-MAPPO v2 Hyperparameters:
  ACTOR_LR            = 0.0003  (FIXED: was 1e-4)
  EPISODES_PER_UPDATE = 16  (FIXED: was 8)
  ENTROPY_COEF        = 0.01  (FIXED: was 0.05)
  Drive save every    = 20 updates


In [11]:
# CELL 11 — Optimizers & Schedulers
actor_opt  = optim.Adam(actor.parameters(),  lr=ACTOR_LR,  eps=1e-5)
critic_opt = optim.Adam(critic.parameters(), lr=CRITIC_LR, eps=1e-5)

timesteps_done = 0

def lr_schedule(t):
    return max(1.0 - t / TOTAL_STEPS, 0.05)

actor_sched  = optim.lr_scheduler.LambdaLR(actor_opt,  lr_lambda=lambda _: lr_schedule(timesteps_done))
critic_sched = optim.lr_scheduler.LambdaLR(critic_opt, lr_lambda=lambda _: lr_schedule(timesteps_done))

obs_norm = RunningNorm(shape=OBS_DIM, clip=5.0)
ret_norm = RunningNorm(shape=1, clip=5.0)
print('Optimizers, schedulers, normalizers ready.')

Optimizers, schedulers, normalizers ready.


In [12]:
# CELL 12 — Rollout Buffer
class Buffer:
    __slots__ = ('obs','states','actions','logprobs','rewards','dones','values')
    def __init__(self):
        for s in self.__slots__: setattr(self, s, [])
    def clear(self):
        for s in self.__slots__: getattr(self, s).clear()

buf = Buffer()
print('Buffer ready.')

Buffer ready.


In [13]:
# CELL 12 — GAE (identical to Phase 1)
def compute_gae(rewards, values, dones):
    adv_list, gae = [], 0.0
    vals = values + [0.0]
    for t in reversed(range(len(rewards))):
        mask  = 1.0 - dones[t]
        delta = rewards[t] + GAMMA * vals[t+1] * mask - vals[t]
        gae   = delta + GAMMA * GAE_LAMBDA * mask * gae
        adv_list.insert(0, gae)
    returns = [a + v for a, v in zip(adv_list, values)]
    return adv_list, returns

def gae_over_buffer():
    all_adv, all_ret = [], []
    ep_start = 0
    for t, d in enumerate(buf.dones):
        if d == 1.0:
            adv, ret = compute_gae(
                buf.rewards[ep_start:t+1],
                buf.values[ep_start:t+1],
                buf.dones[ep_start:t+1]
            )
            all_adv.extend(adv)
            all_ret.extend(ret)
            ep_start = t + 1
    return all_adv, all_ret

In [14]:
# CELL 13 — Action Selection & Value Estimation
# VQC actor has same interface as classical actor — no changes here
@torch.no_grad()
def select_actions(obs_dict):
    actions, logprobs = {}, {}
    for ag in AGENTS:
        obs_norm.update(obs_dict[ag])
        x    = torch.FloatTensor(obs_norm.normalize(obs_dict[ag])).unsqueeze(0)
        dist = Categorical(actor(x))  # actor is now VQCActorNetwork
        a    = dist.sample()
        actions[ag]  = a.item()
        logprobs[ag] = dist.log_prob(a).item()
    return actions, logprobs

@torch.no_grad()
def get_value(obs_dict):
    state = np.concatenate([obs_norm.normalize(obs_dict[ag]) for ag in AGENTS])
    return critic(torch.FloatTensor(state).unsqueeze(0)).item()

def make_global_state(obs_dict):
    return np.concatenate([obs_norm.normalize(obs_dict[ag]) for ag in AGENTS])

In [15]:
# CELL 14 — Collect Episodes (identical to Phase 1)
def collect(n_eps=EPISODES_PER_UPDATE):
    buf.clear()
    raw_ep_rewards = []
    for _ in range(n_eps):
        obs, _ = env.reset()
        ep_raw, done = 0.0, False
        while not done:
            gs               = make_global_state(obs)
            actions, logprbs = select_actions(obs)
            value            = get_value(obs)
            next_obs, rewards, terms, truncs, _ = env.step(actions)
            r_mean = float(np.mean(list(rewards.values())))
            ep_raw += r_mean
            ret_norm.update(np.array([r_mean]))
            r_norm = ret_norm.normalize_scalar(r_mean)
            done = all(terms.values()) or all(truncs.values())
            buf.obs.append(obs); buf.states.append(gs)
            buf.actions.append(actions); buf.logprobs.append(logprbs)
            buf.rewards.append(r_norm); buf.dones.append(float(done))
            buf.values.append(value)
            obs = next_obs
        raw_ep_rewards.append(ep_raw)
    return raw_ep_rewards

test = collect(1)
print(f'Collect test: reward={test[0]:.2f} | buffer={len(buf.rewards)} steps')

Collect test: reward=-31.22 | buffer=50 steps


In [16]:
# CELL 15 — Build Training Tensors (identical to Phase 1)
def build_tensors(advantages, returns):
    states_t  = torch.FloatTensor(np.stack(buf.states))
    returns_t = torch.FloatTensor(returns).unsqueeze(1)
    oldvals_t = torch.FloatTensor(buf.values).unsqueeze(1)
    obs_l, act_l, lp_l, adv_l = [], [], [], []
    for obs_d, act_d, lp_d, adv in zip(buf.obs, buf.actions, buf.logprobs, advantages):
        for ag in AGENTS:
            obs_l.append(obs_norm.normalize(obs_d[ag]))
            act_l.append(act_d[ag])
            lp_l.append(lp_d[ag])
            adv_l.append(adv)
    return (
        torch.FloatTensor(np.stack(obs_l)),
        torch.LongTensor(act_l),
        torch.FloatTensor(lp_l),
        torch.FloatTensor(adv_l),
        states_t, returns_t, oldvals_t
    )

In [17]:
# CELL 16 — Mini-Batch Update (identical to Phase 1 — VQC is fully differentiable)
def minibatch_update(mb_obs, mb_act, mb_lp, mb_adv,
                     mb_states, mb_ret, mb_oldv, actor_update=True):

    mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

    # Classical Critic (unchanged)
    v_pred = critic(mb_states)
    c_loss = VALUE_COEF * nn.functional.mse_loss(v_pred, mb_ret)
    critic_opt.zero_grad()
    c_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), CRITIC_GRAD_CLIP)
    critic_opt.step()

    if not actor_update:
        return 0.0, c_loss.item(), 0.0

    probs   = actor(mb_obs)
    dist    = Categorical(probs)
    new_lp  = dist.log_prob(mb_act)
    entropy = dist.entropy().mean()

    ratio  = torch.exp(new_lp - mb_lp)
    s1     = ratio * mb_adv
    s2     = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * mb_adv
    a_loss = -torch.min(s1, s2).mean() - ENTROPY_COEF * entropy

    actor_opt.zero_grad()
    nn.utils.clip_grad_norm_(actor.parameters(), ACTOR_GRAD_CLIP)
    actor_opt.step()

    return a_loss.item(), c_loss.item(), entropy.item()

In [18]:
# CELL 17 — PPO Epoch Loop (identical to Phase 1)
def ppo_update(tensors, critic_warmup=False):
    actor_obs, actions, old_lp, adv, states, returns, oldvals = tensors
    N = actor_obs.shape[0]
    a_losses, c_losses, entropies = [], [], []

    if critic_warmup:
        for _ in range(CRITIC_WARMUP_STEPS):
            cidx = torch.randperm(states.shape[0])
            for s in range(0, states.shape[0], MINI_BATCH):
                ci = cidx[s:s + MINI_BATCH]
                ai = torch.clamp(ci * NUM_AGENTS, max=N - 1)
                minibatch_update(actor_obs[ai], actions[ai], old_lp[ai], adv[ai],
                                 states[ci], returns[ci], oldvals[ci], actor_update=False)

    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(N)
        for s in range(0, N, MINI_BATCH):
            ai = idx[s:s + MINI_BATCH]
            ci = ai // NUM_AGENTS
            al, cl, ent = minibatch_update(
                actor_obs[ai], actions[ai], old_lp[ai], adv[ai].clone(),
                states[ci], returns[ci], oldvals[ci], actor_update=True)
            a_losses.append(al); c_losses.append(cl); entropies.append(ent)

    return np.mean(a_losses), np.mean(c_losses), np.mean(entropies)

In [19]:
# CELL 18 — Smoke Test
print('Running smoke test...')
ep_rs = collect(EPISODES_PER_UPDATE)
adv, ret = gae_over_buffer()
tens = build_tensors(adv, ret)
al, cl, ent = ppo_update(tens, critic_warmup=False)
print(f'Smoke test PASSED')
print(f'  Mean ep reward : {np.mean(ep_rs):.2f}')
print(f'  Actor loss     : {al:.4f}')
print(f'  Critic loss    : {cl:.4f}')
print(f'  Entropy        : {ent:.4f}')

Running smoke test...
Smoke test PASSED
  Mean ep reward : -62.83
  Actor loss     : -0.0161
  Critic loss    : 262.3974
  Entropy        : 1.6094


In [20]:
# CELL 20 — MAIN TRAINING LOOP with Drive Logging
timesteps_done = 0
best_avg100    = -float('inf')
start_time     = time.time()
all_ep_rewards = []
a_loss_hist    = []
c_loss_hist    = []
ent_hist       = []
update_n       = 0

print(f'Q-MAPPO 200K v2 (Phase 2 Colab) — starting...')
print(f'QI Actor: {N_QUBITS} units, {N_VQC_LAYERS} layers | Critic: 54-dim classical')
print(f'Phase 1 MAPPO 200K baseline: Best Avg100 = -42.08 | Eval Mean = -50.80')
print('-' * 95)

while timesteps_done < TOTAL_STEPS:

    # 1. Collect
    raw_rewards = collect(EPISODES_PER_UPDATE)
    ep_steps    = len(buf.rewards)

    # 2. GAE
    advantages, returns = gae_over_buffer()

    # 3. Build tensors
    tensors = build_tensors(advantages, returns)

    # 4. PPO update
    do_warmup   = (timesteps_done < CRITIC_WARMUP_UNTIL)
    al, cl, ent = ppo_update(tensors, critic_warmup=do_warmup)
    update_n   += 1

    # 5. LR schedulers
    timesteps_done += ep_steps
    actor_sched.step()
    critic_sched.step()

    # 6. Track metrics
    all_ep_rewards.extend(raw_rewards)
    a_loss_hist.append(al)
    c_loss_hist.append(cl)
    ent_hist.append(ent)

    avg100  = float(np.mean(all_ep_rewards[-100:]))
    ep_mean = float(np.mean(raw_rewards))
    elapsed = (time.time() - start_time) / 60

    # 7. Log to CSV
    logger.log({
        'episode'     : len(all_ep_rewards),
        'timestep'    : timesteps_done,
        'ep_reward'   : round(ep_mean, 4),
        'avg100'      : round(avg100, 4),
        'best_avg100' : round(best_avg100, 4),
        'actor_loss'  : round(al, 6),
        'critic_loss' : round(cl, 4),
        'entropy'     : round(ent, 4),
        'actor_lr'    : round(actor_opt.param_groups[0]['lr'], 7),
        'elapsed_min' : round(elapsed, 2)
    })

    # 8. Save best model (local)
    if avg100 > best_avg100:
        best_avg100 = avg100
        torch.save(actor.state_dict(),  f'{LOCAL_DIR}/best_actor.pth')
        torch.save(critic.state_dict(), f'{LOCAL_DIR}/best_critic.pth')
        with open(f'{LOCAL_DIR}/best_obs_norm.pkl', 'wb') as f:
            pickle.dump(obs_norm.save(), f)

    # 9. Save to Drive every DRIVE_SAVE_EVERY updates
    if update_n % DRIVE_SAVE_EVERY == 0:
        logger.save_to_drive()

    # 10. Checkpoint every SAVE_EVERY steps
    if timesteps_done % SAVE_EVERY < ep_steps:
        ckpt_dir = f'{LOCAL_DIR}/checkpoint_{timesteps_done}'
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save(actor.state_dict(),  f'{ckpt_dir}/actor.pth')
        torch.save(critic.state_dict(), f'{ckpt_dir}/critic.pth')
        with open(f'{ckpt_dir}/obs_norm.pkl', 'wb') as f:
            pickle.dump(obs_norm.save(), f)
        # Also copy checkpoint to Drive
        drive_ckpt = f'{DRIVE_DIR}/checkpoint_{timesteps_done}'
        os.makedirs(drive_ckpt, exist_ok=True)
        for fn in os.listdir(ckpt_dir):
            shutil.copy(os.path.join(ckpt_dir, fn), os.path.join(drive_ckpt, fn))
        print(f'\n[Checkpoint {timesteps_done:,}] Best Avg100={best_avg100:.2f} | Saved to Drive\n')

    # 11. Print progress every 5 updates
    if update_n % 5 == 0:
        eta = (elapsed / max(timesteps_done, 1)) * (TOTAL_STEPS - timesteps_done)
        print(
            f'Ep {len(all_ep_rewards):5d} | '
            f'Reward {ep_mean:7.2f} | '
            f'Avg100 {avg100:7.2f} | '
            f'Best {best_avg100:7.2f} | '
            f'Steps {timesteps_done:>7,} | '
            f'Ent {ent:.3f} | '
            f'ALoss {al:.4f} | '
            f'CLoss {cl:.3f} | '
            f'LR {actor_opt.param_groups[0]["lr"]:.5f} | '
            f'ETA {eta:.0f}m'
        )

# Training complete
total_time = time.time() - start_time
torch.save(actor.state_dict(),  f'{LOCAL_DIR}/final_actor.pth')
torch.save(critic.state_dict(), f'{LOCAL_DIR}/final_critic.pth')
with open(f'{LOCAL_DIR}/final_obs_norm.pkl', 'wb') as f: pickle.dump(obs_norm.save(), f)
with open(f'{LOCAL_DIR}/final_ret_norm.pkl', 'wb') as f: pickle.dump(ret_norm.save(), f)

# Final Drive save
logger.save_to_drive()
shutil.copy(f'{LOCAL_DIR}/final_actor.pth',  f'{DRIVE_DIR}/final_actor.pth')
shutil.copy(f'{LOCAL_DIR}/final_critic.pth', f'{DRIVE_DIR}/final_critic.pth')

print(f'\nTraining complete — {total_time/60:.1f} min | Best Avg100: {best_avg100:.2f}')
print(f'Phase 1 MAPPO 200K was   : Best Avg100 = -42.08')
print(f'Q-MAPPO v2 improvement   : {best_avg100 - (-42.08):+.2f}')
print(f'All results saved to Drive: {DRIVE_DIR}')

Q-MAPPO 200K v2 (Phase 2 Colab) — starting...
QI Actor: 8 units, 5 layers | Critic: 54-dim classical
Phase 1 MAPPO 200K baseline: Best Avg100 = -42.08 | Eval Mean = -50.80
-----------------------------------------------------------------------------------------------
Ep    80 | Reward  -62.60 | Avg100  -58.48 | Best  -55.81 | Steps   4,000 | Ent 1.609 | ALoss -0.0161 | CLoss 39.996 | LR 0.00029 | ETA 18m
Ep   160 | Reward  -62.30 | Avg100  -61.71 | Best  -55.81 | Steps   8,000 | Ent 1.609 | ALoss -0.0161 | CLoss 23.605 | LR 0.00029 | ETA 17m
Ep   240 | Reward  -61.35 | Avg100  -61.98 | Best  -55.81 | Steps  12,000 | Ent 1.609 | ALoss -0.0161 | CLoss 8.785 | LR 0.00028 | ETA 16m
  [Drive saved] 08:30:59 -> /content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2
Ep   320 | Reward  -50.81 | Avg100  -57.97 | Best  -55.81 | Steps  16,000 | Ent 1.609 | ALoss -0.0161 | CLoss 10.021 | LR 0.00028 | ETA 16m
Ep   400 | Reward  -56.53 | Avg100  -56.10 | Best  -55.27 | Steps  20,000 | Ent 1.609 | A

In [21]:
# CELL 21 — Final Metrics
print('=' * 60)
print('  Q-MAPPO Phase 2 v2 — 200K — Final Metrics')
print('=' * 60)
print(f'  Mean Reward    : {np.mean(all_ep_rewards):.2f}')
print(f'  Std  Reward    : {np.std(all_ep_rewards):.2f}')
print(f'  Max  Reward    : {np.max(all_ep_rewards):.2f}')
print(f'  Min  Reward    : {np.min(all_ep_rewards):.2f}')
print(f'  Best Avg100    : {best_avg100:.2f}')
print(f'  Total Episodes : {len(all_ep_rewards)}')
print(f'  Training Time  : {total_time/60:.1f} min')
print('=' * 60)
print('  Phase 1 MAPPO 200K:')
print(f'  Best Avg100 = -42.08  |  Eval Mean = -50.80')
print('=' * 60)
logger.summary()

  Q-MAPPO Phase 2 v2 — 200K — Final Metrics
  Mean Reward    : -58.24
  Std  Reward    : 16.26
  Max  Reward    : -18.93
  Min  Reward    : -134.08
  Best Avg100    : -54.02
  Total Episodes : 4000
  Training Time  : 16.5 min
  Phase 1 MAPPO 200K:
  Best Avg100 = -42.08  |  Eval Mean = -50.80
  Total rows logged : 250
  Best Avg100       : -54.02  (ep 3440)
  Log saved to Drive: /content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2/qmappo_v2_log.csv


In [22]:
# CELL 22 — Greedy Evaluation (100 episodes)
@torch.no_grad()
def evaluate(n_eps=100):
    actor.load_state_dict(torch.load(f'{LOCAL_DIR}/best_actor.pth'))
    with open(f'{LOCAL_DIR}/best_obs_norm.pkl', 'rb') as f:
        obs_norm.load(pickle.load(f))
    actor.eval()
    rewards = []
    for _ in range(n_eps):
        obs, _ = env.reset()
        ep_r, done = 0.0, False
        while not done:
            actions = {}
            for ag in AGENTS:
                x = torch.FloatTensor(obs_norm.normalize(obs[ag])).unsqueeze(0)
                actions[ag] = actor(x).argmax(dim=-1).item()
            obs, rews, terms, truncs, _ = env.step(actions)
            ep_r += float(np.mean(list(rews.values())))
            done  = all(terms.values()) or all(truncs.values())
        rewards.append(ep_r)
    actor.train()
    return rewards

eval_rews = evaluate(100)
eval_mean = np.mean(eval_rews)
eval_std  = np.std(eval_rews)

print('=' * 60)
print('  Greedy Evaluation — 100 episodes, best Q-MAPPO v2')
print('=' * 60)
print(f'  Eval Mean : {eval_mean:.2f}')
print(f'  Eval Std  : {eval_std:.2f}')
print(f'  Eval Max  : {np.max(eval_rews):.2f}')
print(f'  Eval Min  : {np.min(eval_rews):.2f}')
print('=' * 60)
print(f'  Phase 1 MAPPO 200K Eval Mean: -50.80')
print(f'  Q-MAPPO v2 improvement: {eval_mean - (-50.80):+.2f}')
print('=' * 60)

# Save eval results to Drive
eval_path = f'{DRIVE_DIR}/eval_results.csv'
with open(eval_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['episode', 'reward'])
    for i, r in enumerate(eval_rews):
        w.writerow([i+1, round(r, 4)])
print(f'Eval results saved: {eval_path}')

  Greedy Evaluation — 100 episodes, best Q-MAPPO v2
  Eval Mean : -49.46
  Eval Std  : 16.71
  Eval Max  : -18.69
  Eval Min  : -102.83
  Phase 1 MAPPO 200K Eval Mean: -50.80
  Q-MAPPO v2 improvement: +1.34
Eval results saved: /content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2/eval_results.csv


In [23]:
# CELL 23 — Training Curves + Save to Drive
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Q-MAPPO Phase 2 v2 — 200K Steps | Best Avg100={best_avg100:.2f}', fontsize=13)

ax = axes[0, 0]
ax.plot(all_ep_rewards, alpha=0.2, color='steelblue', lw=0.5)
w = 100
if len(all_ep_rewards) >= w:
    ma = np.convolve(all_ep_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w-1, len(all_ep_rewards)), ma, color='steelblue', lw=2,
            label=f'Q-MAPPO v2 Avg100 (best={best_avg100:.1f})')
ax.axhline(best_avg100, color='green',  ls='--', alpha=0.6, label=f'Best={best_avg100:.1f}')
ax.axhline(-42.08,      color='red',    ls=':',  alpha=0.7, label='Phase1 baseline=-42.08')
ax.set_title('Episode Reward'); ax.set_xlabel('Episode')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(a_loss_hist, color='coral', alpha=0.7)
ax.set_title('Actor Loss'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.semilogy(np.maximum(np.array(c_loss_hist), 1e-6), color='mediumseagreen', alpha=0.7)
ax.set_title('Critic Loss (log)'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(ent_hist, color='mediumpurple', alpha=0.8)
ax.axhline(0.3, color='red',   ls='--', alpha=0.5, label='min target')
ax.axhline(1.6, color='green', ls='--', alpha=0.4, label='uniform (1.6)')
ax.set_title('Policy Entropy (nats)'); ax.set_xlabel('Update'); ax.set_ylim(0, 1.8)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_local = f'{LOCAL_DIR}/qmappo_v2_200K_curves.png'
plot_drive = f'{DRIVE_DIR}/qmappo_v2_200K_curves.png'
plt.savefig(plot_local, dpi=150, bbox_inches='tight')
shutil.copy(plot_local, plot_drive)
plt.show()
print(f'Plot saved to Drive: {plot_drive}')

Plot saved to Drive: /content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2/qmappo_v2_200K_curves.png


In [24]:
# CELL 24 — Final Summary + Zip All Results
print('=' * 65)
print('  Q-MAPPO Phase 2 v2 — 200K — COMPLETE')
print('=' * 65)
print(f'  Best Avg100      : {best_avg100:.2f}  (Phase1 = -42.08)')
print(f'  Eval Mean        : {np.mean(eval_rews):.2f}  (Phase1 = -50.80)')
print(f'  Eval Std         : {np.std(eval_rews):.2f}')
print(f'  Training Time    : {total_time/60:.1f} min')
print(f'  Actor Params     : {sum(p.numel() for p in actor.parameters())} (99.4% fewer than Phase1)')
print(f'  Improvement      : {best_avg100 - (-42.08):+.2f} Best Avg100 | {np.mean(eval_rews)-(-50.80):+.2f} Eval Mean')
print('=' * 65)
print(f'  All files in Drive: {DRIVE_DIR}')
print('  Files saved:')
for f in sorted(os.listdir(DRIVE_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) if os.path.isfile(os.path.join(DRIVE_DIR, f)) else 0
    print(f'    {f:45s} {size/1024:.1f} KB')

# Create zip of all local results
zip_local = f'{LOCAL_DIR}/QMAPPO_200K_v2.zip'
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            if file.endswith('.zip'): continue
            full = os.path.join(root, file)
            zf.write(full, os.path.relpath(full, LOCAL_DIR))
zip_drive = f'{DRIVE_DIR}/QMAPPO_200K_v2.zip'
shutil.copy(zip_local, zip_drive)
print(f'\n  Zip: {zip_drive} ({os.path.getsize(zip_drive)/1e6:.1f} MB)')
print('  Download from Files panel or from Google Drive.')

  Q-MAPPO Phase 2 v2 — 200K — COMPLETE
  Best Avg100      : -54.02  (Phase1 = -42.08)
  Eval Mean        : -49.46  (Phase1 = -50.80)
  Eval Std         : 16.71
  Training Time    : 16.5 min
  Actor Params     : 1357 (99.4% fewer than Phase1)
  Improvement      : -11.94 Best Avg100 | +1.34 Eval Mean
  All files in Drive: /content/drive/MyDrive/MTech_MARL_Phase2/QMAPPO_200K_v2
  Files saved:
    best_actor.pth                                13.3 KB
    best_critic.pth                               1665.1 KB
    best_obs_norm.pkl                             0.5 KB
    checkpoint_100000                             0.0 KB
    checkpoint_150400                             0.0 KB
    checkpoint_200000                             0.0 KB
    checkpoint_50400                              0.0 KB
    eval_results.csv                              1.3 KB
    final_actor.pth                               13.3 KB
    final_critic.pth                              1665.1 KB
    qmappo_v2_200K_curves.png